# MLP Ultra96 — AXI DMA dataset runner

This notebook runs a quantized MLP bitstream on the Ultra96 through PYNQ and AXI DMA.
It sends quantized `int16` input samples to the FPGA, reads back `int16` predictions,
aligns the results according to the selected FPGA architecture, and saves a CSV report.

The notebook is organized as a hardware test procedure:
1. select the FPGA mode and discover project files;
2. load helper functions and the quantized dataset;
3. open the PYNQ overlay and select the DMA IP;
4. stream the dataset through the FPGA;
5. compare FPGA outputs with dataset references;
6. export an aligned CSV report.

Only change the architecture selector near the beginning of the notebook:

```python
FPGA_MODE = "sequential"
# or
FPGA_MODE = "pipeline"
```

The rest of the notebook derives the required settings from the dataset and `scales.vh` when available.
Run the cells from top to bottom after changing the mode, dataset, or bitstream.


## 1. Configuration and project-file discovery

This first code cell contains the user-facing settings. In most cases, only `FPGA_MODE`, the bitstream location, or the dataset location need to be changed.

The cell also tries to infer useful metadata from `scales.vh`, especially `NUM_LINEAR`, `N0`, and the output size. These values are later used to align pipeline outputs and to document the CSV report.


In [ ]:
from pathlib import Path
import re
import time
import numpy as np
import pandas as pd

from pynq import Overlay, allocate

# ============================================================
# User configuration
# ============================================================

# FPGA design selection.
# Allowed values: "sequential" or "pipeline".
FPGA_MODE = "sequential"

BASE = Path.cwd()

# Bitstream discovery.
# Priority:
#   1) design_1.bit
#   2) TOP_SOC_MLP.bit
#   3) first .bit file in the current directory
BITFILE_CANDIDATES = [
    BASE / "design_1.bit",
    BASE / "TOP_SOC_MLP.bit",
]
BITFILE = next((p for p in BITFILE_CANDIDATES if p.exists()), None)
if BITFILE is None:
    BITFILE = next(BASE.glob("*.bit"), None)

# Quantized dataset generated by export_quantized_dataset_npz().
# Priority:
#   1) dataset_walter_quantized.npz
#   2) alternate MLP_dataset_ideal_quantized_int16.npz filename
#   3) first .npz file in the current directory
PREFERRED_NPZ = BASE / "dataset_walter_quantized.npz"
LEGACY_NPZ = BASE / "MLP_dataset_ideal_quantized_int16.npz"

if PREFERRED_NPZ.exists():
    DATASET_NPZ = PREFERRED_NPZ
elif LEGACY_NPZ.exists():
    DATASET_NPZ = LEGACY_NPZ
else:
    DATASET_NPZ = next(BASE.glob("*.npz"), None)

DMA_NAME = "axi_dma_0"
OUT_DIR = BASE / "Result"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional scales header copied with the bitstream/hwh.
# Used to infer NUM_LINEAR for pipeline alignment and CSV metadata.
SCALES_VH = BASE / "scales.vh"

# Use a small value for quick debugging, or None to test the full dataset.
MAX_SAMPLES = None

# Pipeline alignment configuration.
# If scales.vh is present, NUM_LINEAR is read automatically.
# Otherwise, NUM_LINEAR_FALLBACK is used.
NUM_LINEAR_FALLBACK = 6
FLUSH_VALUE = 0

# Overlay behavior.
GEN_CACHE = False

# ============================================================
# Input checks and mode-derived parameters
# ============================================================

FPGA_MODE = FPGA_MODE.lower().strip()
if FPGA_MODE not in {"sequential", "pipeline"}:
    raise ValueError("FPGA_MODE must be either 'sequential' or 'pipeline'.")

if BITFILE is None:
    raise FileNotFoundError("No .bit file found in the current folder.")
if DATASET_NPZ is None:
    raise FileNotFoundError("No .npz dataset found in the current folder.")

def read_num_linear_from_scales(scales_path: Path, default: int) -> int:
    if not scales_path.exists():
        return int(default)
    txt = scales_path.read_text(encoding="utf-8", errors="ignore")
    m = re.search(r"`define\s+NUM_LINEAR\s+(\d+)", txt)
    return int(m.group(1)) if m else int(default)

if FPGA_MODE == "pipeline":
    NUM_LINEAR = read_num_linear_from_scales(SCALES_VH, NUM_LINEAR_FALLBACK)
    WARMUP_FRAMES = NUM_LINEAR - 1
    NUM_FLUSH_SAMPLES = WARMUP_FRAMES
else:
    NUM_LINEAR = read_num_linear_from_scales(SCALES_VH, NUM_LINEAR_FALLBACK)
    WARMUP_FRAMES = 0
    NUM_FLUSH_SAMPLES = 0

print("BASE              =", BASE)
print("FPGA_MODE         =", FPGA_MODE)
print("BITFILE           =", BITFILE)
print("DATASET           =", DATASET_NPZ)
print("OUT_DIR           =", OUT_DIR)
print("SCALES_VH         =", SCALES_VH if SCALES_VH.exists() else "not found")
print("NUM_LINEAR        =", NUM_LINEAR)
print("WARMUP_FRAMES     =", WARMUP_FRAMES)
print("NUM_FLUSH_SAMPLES =", NUM_FLUSH_SAMPLES)
print("DMA_NAME          =", DMA_NAME)

## 2. Shared helpers

The helper functions below keep the rest of the notebook readable. They handle three recurring tasks:
- preserving signed `int16` values while transporting them as 16-bit AXI words;
- selecting the correct arrays from the exported `.npz` dataset;
- converting quantized integer values back to floating-point values for reporting.

These functions do not access the FPGA yet.


In [ ]:
# ============================================================
# int16 and AXI 16-bit helpers
# ============================================================

def int16_to_uint16_words(arr: np.ndarray) -> np.ndarray:
    """Prepare int16 values for a 16-bit AXI stream without changing the bits."""
    return np.asarray(arr, dtype=np.int16).reshape(-1).view(np.uint16)


def uint16_words_to_int16(arr: np.ndarray) -> np.ndarray:
    """Convert received 16-bit AXI words back to signed int16 values."""
    return np.asarray(arr, dtype=np.uint16).reshape(-1).view(np.int16)


def pick_npz_array(ds, preferred_names, ndim=None, required=True):
    """Return the first matching NPZ array from the preferred key list."""
    for name in preferred_names:
        if name in ds.files:
            arr = ds[name]
            if ndim is None or arr.ndim == ndim:
                return name, arr
    if required:
        raise KeyError(
            f"Missing key in NPZ. Searched: {preferred_names}. "
            f"Available keys: {list(ds.files)}"
        )
    return None, None


def get_npz_scalar(ds, names, default=None):
    """Read a scalar value from the NPZ file if one of the keys exists."""
    for name in names:
        if name in ds.files:
            return int(np.asarray(ds[name]).reshape(-1)[0])
    return default


def q_to_float(x_int: np.ndarray, scale_exp: int) -> np.ndarray:
    """Convert fixed-point int values to float using q / 2**scale_exp."""
    return np.asarray(x_int, dtype=np.float64) / float(1 << int(scale_exp))

## 3. Load the quantized dataset

This cell loads the dataset exported by the quantization notebook. It expects quantized inputs and targets, and optionally stores float references and PyTorch prediction references.

The FPGA always receives the quantized input array. Float arrays are only used to make the final CSV easier to inspect.


In [ ]:
# ============================================================
# Load the quantized NPZ dataset
# ============================================================

ds = np.load(DATASET_NPZ)
print("Available NPZ keys:", list(ds.files))

# Preferred keys produced by the quantization export.
X_FLOAT_KEY, X_float = pick_npz_array(ds, ["ph_val", "x_float", "X_float", "x"], ndim=2, required=False)
X_INT_KEY, X = pick_npz_array(ds, ["ph_val_int", "X_int", "x_int", "inputs_int", "input_int"], ndim=2, required=False)

Y_FLOAT_KEY, Y_float = pick_npz_array(ds, ["pos", "y_float", "Y_float", "target_float"], ndim=2, required=False)
Y_INT_KEY, Y_dataset = pick_npz_array(ds, ["pos_int", "Y_int", "y_int", "targets_int", "output_int"], ndim=2, required=False)

PRED_FLOAT_KEY, Y_pred_float_ref = pick_npz_array(ds, ["pred_float", "float", "prediction_float"], ndim=2, required=False)
PRED_INT_KEY, Y_pred_int_ref = pick_npz_array(ds, ["pred_int", "float_int", "prediction_int"], ndim=2, required=False)

# Fallback: some datasets store the int16 arrays directly under generic keys.
# This keeps the notebook usable with compact exported datasets.
if X is None:
    legacy_x_key, legacy_x = pick_npz_array(ds, ["ph_val", "X", "x", "inputs", "input", "features"], ndim=2)
    X_INT_KEY, X = legacy_x_key, legacy_x.astype(np.int16)

if Y_dataset is None:
    legacy_y_key, legacy_y = pick_npz_array(ds, ["pos", "Y", "y", "outputs", "output", "labels", "targets"], ndim=2)
    Y_INT_KEY, Y_dataset = legacy_y_key, legacy_y.astype(np.int16)

X = X.astype(np.int16)
Y_dataset = Y_dataset.astype(np.int16)

if Y_pred_int_ref is not None:
    Y_pred_int_ref = Y_pred_int_ref.astype(np.int16)

input_scale_exp = get_npz_scalar(ds, ["input_scale_exp", "input_exp", "x_scale_exp"])
target_scale_exp = get_npz_scalar(ds, ["target_scale_exp", "target_exp", "output_scale_exp", "y_scale_exp"])
pred_scale_exp = get_npz_scalar(ds, ["pred_scale_exp", "prediction_scale_exp", "float_scale_exp"], default=target_scale_exp)

# Use one scale for all output-side fixed-point values.
# In the current export, target and prediction outputs use the same exponent.
output_scale_exponent = pred_scale_exp if pred_scale_exp is not None else target_scale_exp

N_SAMPLES, INPUT_SIZE = X.shape
N_Y, OUTPUT_SIZE = Y_dataset.shape

assert N_Y == N_SAMPLES, "Inputs and targets must have the same number of samples."

print("X key              =", X_INT_KEY, X.shape, X.dtype)
print("Y target key       =", Y_INT_KEY, Y_dataset.shape, Y_dataset.dtype)
print("X float key        =", X_FLOAT_KEY)
print("Y float key        =", Y_FLOAT_KEY)
print("Pred float key     =", PRED_FLOAT_KEY)
print("Pred int key       =", PRED_INT_KEY)
print("input_scale_exp    =", input_scale_exp)
print("target_scale_exp   =", target_scale_exp)
print("pred_scale_exp     =", pred_scale_exp)
print("output_scale_exp   =", output_scale_exponent)
print("N_SAMPLES          =", N_SAMPLES)
print("INPUT_SIZE         =", INPUT_SIZE)
print("OUTPUT_SIZE        =", OUTPUT_SIZE)

## 4. Load the Ultra96 overlay and select the DMA

This cell programs the FPGA with the selected bitstream and retrieves the AXI DMA IP from the PYNQ overlay.

If the DMA name changed during Vivado block-design generation, the error message prints the available IP names so the notebook can be updated without guessing.


In [ ]:
# ============================================================
# Load the PYNQ overlay and select the DMA IP
# ============================================================

ol = Overlay(str(BITFILE), gen_cache=GEN_CACHE)

print("Available IPs in the overlay:")
for name in ol.ip_dict.keys():
    print(" -", name)

if not hasattr(ol, DMA_NAME):
    dma_candidates = [name for name in ol.ip_dict.keys() if "dma" in name.lower()]
    raise RuntimeError(
        f"DMA_NAME='{DMA_NAME}' not found. "
        f"Possible candidates: {dma_candidates}"
    )

dma = getattr(ol, DMA_NAME)
print("DMA used:", DMA_NAME)

## 5. Stream the dataset through the FPGA

This is the hardware execution step. Each input sample is packed as signed `int16` words, sent through DMA MM2S, and the FPGA response is read back through DMA S2MM.

In `pipeline` mode, the first outputs are warmup frames and extra flush frames are sent at the end. In `sequential` mode, each received frame is aligned directly with the input sample that produced it.


In [ ]:
# ============================================================
# Run the FPGA design through AXI DMA
# The transaction loop below adapts the output alignment to FPGA_MODE.
# ============================================================
#
# AXI stream protocol used by the generated Ultra96 RTL:
#   - AXI-Stream transport uses 16-bit words
#   - signed int16 values are sent without changing the bit pattern
#   - on the RTL input side, the useful DW bits are taken from the LSBs
#   - on the RTL output side, sign extension is applied from DW to 16 bits
#
# Debug information saved later in the CSV:
#   - the raw output frame used for each sample
#   - TX/RX words in hexadecimal
#   - the AXI transport protocol name

N_TEST = N_SAMPLES if MAX_SAMPLES is None else min(int(MAX_SAMPLES), N_SAMPLES)

TX_WORDS_PER_SAMPLE = INPUT_SIZE
RX_WORDS_PER_SAMPLE = OUTPUT_SIZE

tx_buf = allocate(shape=(TX_WORDS_PER_SAMPLE,), dtype=np.uint16)
rx_buf = allocate(shape=(RX_WORDS_PER_SAMPLE,), dtype=np.uint16)

print("Starting board execution...")
print("FPGA_MODE           =", FPGA_MODE)
print("N_TEST              =", N_TEST)
print("TX_WORDS_PER_SAMPLE =", TX_WORDS_PER_SAMPLE)
print("RX_WORDS_PER_SAMPLE =", RX_WORDS_PER_SAMPLE)
print("AXI protocol        = 16-bit signed words, no SOF/EOF, frame size fixed by INPUT_SIZE/OUTPUT_SIZE")

def words_to_hex(words: np.ndarray) -> str:
    """Compact hex dump of uint16 AXI words."""
    return " ".join(f"{int(w) & 0xFFFF:04x}" for w in np.asarray(words, dtype=np.uint16).reshape(-1))

t0 = time.time()

if FPGA_MODE == "sequential":
    # ------------------------------------------------------------
    # Sequential design:
    #   each input sample produces its output in the same transaction.
    # ------------------------------------------------------------
    Y_pred_board = np.zeros((N_TEST, OUTPUT_SIZE), dtype=np.int16)

    tx_words_hex_all = []
    rx_words_hex_all = []
    raw_output_frame_all = np.arange(N_TEST, dtype=np.int32)

    for i in range(N_TEST):
        tx_words = int16_to_uint16_words(X[i]).copy()

        tx_buf[:] = tx_words
        rx_buf[:] = 0

        tx_buf.flush()
        rx_buf.flush()

        # Arm recv first, then send.
        dma.recvchannel.transfer(rx_buf)
        dma.sendchannel.transfer(tx_buf)

        dma.sendchannel.wait()
        dma.recvchannel.wait()

        rx_buf.invalidate()

        rx_words = np.asarray(rx_buf, dtype=np.uint16).copy()
        Y_pred_board[i, :] = uint16_words_to_int16(rx_words).copy()

        tx_words_hex_all.append(words_to_hex(tx_words))
        rx_words_hex_all.append(words_to_hex(rx_words))

        if (i + 1) % 10 == 0 or (i + 1) == N_TEST:
            print(f"  {i + 1}/{N_TEST} samples processed")

    Y_stream_board = None
    Y_warmup_invalid = None
    Y_flush_tail = None

else:
    # ------------------------------------------------------------
    # Pipeline design:
    #   each input sample produces its aligned output after WARMUP_FRAMES frames.
    # ------------------------------------------------------------
    NUM_STREAM_SAMPLES = N_TEST + NUM_FLUSH_SAMPLES

    X_real = X[:N_TEST]
    X_flush = np.full((NUM_FLUSH_SAMPLES, INPUT_SIZE), FLUSH_VALUE, dtype=np.int16)
    X_stream = np.vstack([X_real, X_flush]).astype(np.int16)

    assert X_stream.shape == (NUM_STREAM_SAMPLES, INPUT_SIZE)

    Y_stream_board = np.zeros((NUM_STREAM_SAMPLES, OUTPUT_SIZE), dtype=np.int16)

    raw_tx_words_hex = []
    raw_rx_words_hex = []

    print("NUM_LINEAR          =", NUM_LINEAR)
    print("WARMUP_FRAMES       =", WARMUP_FRAMES)
    print("NUM_FLUSH_SAMPLES   =", NUM_FLUSH_SAMPLES)
    print("NUM_STREAM_SAMPLES  =", NUM_STREAM_SAMPLES)

    for frame in range(NUM_STREAM_SAMPLES):
        tx_words = int16_to_uint16_words(X_stream[frame]).copy()

        tx_buf[:] = tx_words
        rx_buf[:] = 0

        tx_buf.flush()
        rx_buf.flush()

        # Short AXI DMA transaction:
        #   RX is armed before TX, then one input frame and one output frame are transferred.
        dma.recvchannel.transfer(rx_buf)
        dma.sendchannel.transfer(tx_buf)

        dma.recvchannel.wait()
        dma.sendchannel.wait()

        rx_buf.invalidate()

        rx_words = np.asarray(rx_buf, dtype=np.uint16).copy()
        Y_stream_board[frame, :] = uint16_words_to_int16(rx_words).copy()

        raw_tx_words_hex.append(words_to_hex(tx_words))
        raw_rx_words_hex.append(words_to_hex(rx_words))

        if frame < WARMUP_FRAMES:
            print(f"  frame {frame + 1}/{NUM_STREAM_SAMPLES} received <warmup/invalid>: {Y_stream_board[frame]}")
        elif frame < WARMUP_FRAMES + N_TEST:
            sample_idx = frame - WARMUP_FRAMES
            if (sample_idx + 1) % 1000 == 0 or (sample_idx + 1) == N_TEST:
                print(f"  {sample_idx + 1}/{N_TEST} real samples aligned")
        else:
            print(f"  frame {frame + 1}/{NUM_STREAM_SAMPLES} received <flush tail>: {Y_stream_board[frame]}")

    Y_pred_board = Y_stream_board[WARMUP_FRAMES:WARMUP_FRAMES + N_TEST].copy()
    Y_warmup_invalid = Y_stream_board[:WARMUP_FRAMES].copy()
    Y_flush_tail = Y_stream_board[WARMUP_FRAMES + N_TEST:].copy()

    raw_output_frame_all = np.arange(WARMUP_FRAMES, WARMUP_FRAMES + N_TEST, dtype=np.int32)
    tx_words_hex_all = raw_tx_words_hex[:N_TEST]
    rx_words_hex_all = raw_rx_words_hex[WARMUP_FRAMES:WARMUP_FRAMES + N_TEST]

t1 = time.time()

print("Execution completed.")
print(f"Total time: {t1 - t0:.3f} s")
if FPGA_MODE == "pipeline":
    print(f"Average time per stream frame: {(t1 - t0) / max(N_TEST + NUM_FLUSH_SAMPLES, 1) * 1000:.3f} ms/frame")
print(f"Average time per real sample: {(t1 - t0) / max(N_TEST, 1) * 1000:.3f} ms/sample")
print("Y_pred_board shape =", Y_pred_board.shape)

if FPGA_MODE == "pipeline":
    print("Y_stream_board shape =", Y_stream_board.shape)
    print("Warmup invalid shape =", Y_warmup_invalid.shape)
    print("Flush tail shape     =", Y_flush_tail.shape)


## 6. Compare FPGA outputs with references

After alignment, this cell compares the FPGA output against the quantized target dataset and, when available, against the quantized reference prediction generated by the floating-point model.

The target comparison checks task accuracy against the dataset. The prediction comparison is usually the best way to validate the fixed-point FPGA implementation against the exported model.


In [ ]:
# ============================================================
# Compare FPGA outputs with dataset references
# ============================================================

Y_ref_target = Y_dataset[:N_TEST]
err_vs_target = Y_pred_board.astype(np.int32) - Y_ref_target.astype(np.int32)

print("FPGA_MODE                =", FPGA_MODE)
print("Y_pred_board shape       =", Y_pred_board.shape)
print("Target reference shape   =", Y_ref_target.shape)

if FPGA_MODE == "pipeline":
    print("Warmup frames ignored    =", WARMUP_FRAMES)

print("Max abs error vs target  =", np.max(np.abs(err_vs_target)))
print("Mean abs error vs target =", np.mean(np.abs(err_vs_target)))
print("Exact samples vs target  =", np.sum(np.all(err_vs_target == 0, axis=1)), "/", N_TEST)

if Y_pred_int_ref is not None:
    Y_ref_pred_int = Y_pred_int_ref[:N_TEST]
    err_vs_pred_int = Y_pred_board.astype(np.int32) - Y_ref_pred_int.astype(np.int32)
    print("------------------------------------------------------------")
    print("Pred int reference shape  =", Y_ref_pred_int.shape)
    print("Max abs error vs pred_int =", np.max(np.abs(err_vs_pred_int)))
    print("Mean abs error vs pred_int=", np.mean(np.abs(err_vs_pred_int)))
    print("Exact samples vs pred_int =", np.sum(np.all(err_vs_pred_int == 0, axis=1)), "/", N_TEST)
else:
    Y_ref_pred_int = None
    err_vs_pred_int = None
    print("------------------------------------------------------------")
    print("No pred_int reference found in the dataset.")

preview_n = min(10, N_TEST)
for i in range(preview_n):
    if FPGA_MODE == "pipeline":
        out_frame = WARMUP_FRAMES + i
        line = (
            f"sample {i:04d} | out_frame={out_frame:04d} | "
            f"target_int={Y_ref_target[i]} | fpga={Y_pred_board[i]} | err_target={err_vs_target[i]}"
        )
    else:
        line = f"sample {i:04d} | target_int={Y_ref_target[i]} | fpga={Y_pred_board[i]} | err_target={err_vs_target[i]}"

    if Y_ref_pred_int is not None:
        line += f" | pred_int={Y_ref_pred_int[i]} | err_pred={err_vs_pred_int[i]}"
    print(line)

## 7. Export the aligned CSV report

The final CSV is designed for debugging and cross-platform comparison. It keeps the original sample index, the raw output frame used after alignment, all float/int references, FPGA outputs, quantized errors, and AXI transport metadata.

The `output_scale_exponent` column is the scale used to reconstruct `fpga_float_*` from the quantized FPGA outputs.


In [ ]:
# ============================================================
# Save the aligned CSV report
# ============================================================
#
# Column order:
#   1) sample
#   2) raw_output_frame, useful for pipeline alignment
#   3) input float
#   4) input quantized int
#   5) target float
#   6) target quantized int
#   7) PyTorch/float-model prediction float
#   8) PyTorch/float-model prediction quantized int
#   9) FPGA prediction quantized int
#  10) FPGA prediction float
#  11) quantized errors
#  12) AXI transport/debug metadata
#
# Notes:
# - AXI transport is fixed to signed 16-bit words.
# - rx_words_hex contains the aligned output frame after warmup removal in pipeline mode.
# - fpga_float and the CSV output_scale_exponent column use the same output-side scale.

cols = {"sample": np.arange(N_TEST, dtype=np.int32)}

# ------------------------------------------------------------
# Output-frame alignment information
# ------------------------------------------------------------

cols["raw_output_frame"] = raw_output_frame_all.astype(np.int32)

# ------------------------------------------------------------
# Inputs
# ------------------------------------------------------------

if X_float is not None:
    for j in range(INPUT_SIZE):
        cols[f"input_float_{j}"] = X_float[:N_TEST, j].astype(np.float32)

for j in range(INPUT_SIZE):
    cols[f"input_quantized_{j}"] = X[:N_TEST, j].astype(np.int32)

# ------------------------------------------------------------
# Targets
# ------------------------------------------------------------

if Y_float is not None:
    for j in range(OUTPUT_SIZE):
        cols[f"target_float_{j}"] = Y_float[:N_TEST, j].astype(np.float32)

for j in range(OUTPUT_SIZE):
    cols[f"target_quantized_{j}"] = Y_ref_target[:, j].astype(np.int32)

# ------------------------------------------------------------
# Reference predictions from the float/PyTorch model
# ------------------------------------------------------------

if Y_pred_float_ref is not None:
    for j in range(OUTPUT_SIZE):
        cols[f"pred_float_{j}"] = Y_pred_float_ref[:N_TEST, j].astype(np.float32)

if Y_ref_pred_int is not None:
    for j in range(OUTPUT_SIZE):
        cols[f"pred_quantized_{j}"] = Y_ref_pred_int[:, j].astype(np.int32)

# ------------------------------------------------------------
# FPGA predictions
# ------------------------------------------------------------

for j in range(OUTPUT_SIZE):
    cols[f"fpga_quantized_{j}"] = Y_pred_board[:, j].astype(np.int32)

# Real-valued FPGA output reconstructed from int16.
# Use the same output scale stored later in the CSV metadata.
if output_scale_exponent is not None:
    fpga_float = q_to_float(Y_pred_board, output_scale_exponent)
    for j in range(OUTPUT_SIZE):
        cols[f"fpga_float_{j}"] = fpga_float[:, j].astype(np.float32)

# ------------------------------------------------------------
# Quantized errors
# ------------------------------------------------------------

for j in range(OUTPUT_SIZE):
    cols[f"error_fpga_target_quantized_{j}"] = err_vs_target[:, j].astype(np.int32)

if err_vs_pred_int is not None:
    for j in range(OUTPUT_SIZE):
        cols[f"error_fpga_pred_quantized_{j}"] = err_vs_pred_int[:, j].astype(np.int32)

# ------------------------------------------------------------
# AXI transport and debug metadata
# ------------------------------------------------------------

cols["tx_words_hex"] = np.asarray(tx_words_hex_all, dtype=object)
cols["rx_words_hex"] = np.asarray(rx_words_hex_all, dtype=object)
cols["transport"] = np.full(N_TEST, "axi_stream_int16_no_sof_send_then_read", dtype=object)
cols["fpga_mode"] = np.full(N_TEST, FPGA_MODE, dtype=object)
cols["num_linear"] = np.full(N_TEST, NUM_LINEAR, dtype=np.int32)
cols["pipeline_delay"] = np.full(N_TEST, WARMUP_FRAMES, dtype=np.int32)
cols["input_scale_exponent"] = np.full(
    N_TEST,
    -1 if input_scale_exp is None else input_scale_exp,
    dtype=np.int32,
)
cols["output_scale_exponent"] = np.full(
    N_TEST,
    -1 if output_scale_exponent is None else output_scale_exponent,
    dtype=np.int32,
)

df = pd.DataFrame(cols)

csv_path = OUT_DIR / f"soc_mlp_fpga_{FPGA_MODE}_axi_results.csv"
df.to_csv(csv_path, index=False)

print("CSV saved:", csv_path)
print("Columns:")
for col in df.columns:
    print(" -", col)

display(df.head(50))


## End

The generated CSV is written to `Result/` and includes:
- aligned FPGA predictions;
- target and optional reference-model predictions;
- quantized errors;
- AXI TX/RX words in hexadecimal;
- mode metadata such as `fpga_mode`, `num_linear`, and `pipeline_delay`.

To switch the FPGA architecture, only edit:

```python
FPGA_MODE = "sequential"
```

or:

```python
FPGA_MODE = "pipeline"
```

Then rerun the notebook from the beginning so all derived variables are recomputed.
